# Enhanced Barra Loader Testing

This notebook tests the enhanced Barra loader with robust InfluxDB connectivity and model maintenance capabilities.

In [ ]:
import sys
import os
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

# Add src directory to path
sys.path.append('/home/jovyan/work/src')

# Import our enhanced Barra loader
from barra_loader import EnhancedBarraLoader

In [ ]:
# Initialize the enhanced Barra loader
print("Initializing Enhanced Barra Loader...")
loader = EnhancedBarraLoader()

# Check connection status
status = loader.get_connection_status()
print("\nConnection Status:")
for key, value in status.items():
    print(f"  {key}: {value}")

In [ ]:
# Test the fixes for InfluxDB v3 protocol issues
print("Testing fixes for InfluxDB v3 protocol...")

# Reload the module to get the latest fixes
import importlib
import barra_loader
importlib.reload(barra_loader)
from barra_loader import EnhancedBarraLoader

# Initialize with updated loader
print("Reinitializing Enhanced Barra Loader with fixes...")
loader_fixed = EnhancedBarraLoader()

# Check connection status with fixes
status_fixed = loader_fixed.get_connection_status()
print("\nFixed Connection Status:")
for key, value in status_fixed.items():
    print(f"  {key}: {value}")

# Test if queries work now
if status_fixed.get('client_initialized', False):
    print("\nTesting basic query functionality...")
    try:
        # Try loading a small amount of recent data
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
        
        print(f"Attempting to load data from {start_date} to {end_date}")
        test_data = loader_fixed.load_factor_data(start_date, end_date)
        
        if not test_data.empty:
            print(f"✓ SUCCESS: Loaded {len(test_data)} records")
            print("Sample data:")
            print(test_data.head(2))
        else:
            print("⚠ No data returned, but query executed without major errors")
            
    except Exception as e:
        print(f"✗ Query still failing: {e}")
else:
    print("Client initialization failed")

In [ ]:
# Test InfluxDB v2 corrected implementation
print("Testing InfluxDB v2 corrected implementation...")

# Reload the module to get the latest fixes
import importlib
import barra_loader
importlib.reload(barra_loader)
from barra_loader import EnhancedBarraLoader

# Initialize with corrected v2 client
print("Initializing Enhanced Barra Loader with InfluxDB v2 client...")
loader_v2 = EnhancedBarraLoader()

# Check connection status
status_v2 = loader_v2.get_connection_status()
print("\nInfluxDB v2 Connection Status:")
for key, value in status_v2.items():
    print(f"  {key}: {value}")

# Test health check and basic functionality
if status_v2.get('client_initialized', False):
    print(f"\n✓ InfluxDB client version: {loader_v2.influx_version}")
    print("✓ Connection initialized successfully")
    
    # Test loading data
    print("\nTesting data loading with InfluxDB v2...")
    try:
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = (datetime.now() - timedelta(days=5)).strftime('%Y-%m-%d')
        
        print(f"Loading data from {start_date} to {end_date}")
        test_data = loader_v2.load_factor_data(start_date, end_date)
        
        if not test_data.empty:
            print(f"✓ SUCCESS: Loaded {len(test_data)} records")
            print("\nColumns available:")
            print(test_data.columns.tolist())
            print("\nFirst few rows:")
            print(test_data.head(2))
        else:
            print("⚠ No data returned - may be no data in date range")
    except Exception as e:
        print(f"✗ Data loading failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print("✗ Client initialization failed")

In [ ]:
# Simple InfluxDB v2 connectivity test
print("Testing basic InfluxDB v2 connectivity...")

try:
    from influxdb_client import InfluxDBClient
    
    # Test direct connection
    url = "http://influxdb:8086"
    token = "qtab_factor_token_2024_secure"
    org = "qtab"
    
    print(f"Connecting to {url}")
    client = InfluxDBClient(url=url, token=token, org=org)
    
    # Test health
    health = client.health()
    print(f"Health status: {health.status}")
    print(f"Health message: {health.message}")
    
    if health.status == "pass":
        print("✓ InfluxDB v2 connection successful!")
        
        # Test query API
        query_api = client.query_api()
        
        # Simple bucket list query
        buckets_query = '''
        buckets()
            |> filter(fn: (r) => r.name != "_monitoring" and r.name != "_tasks")
            |> limit(n: 10)
        '''
        
        print("Testing query API...")
        result = query_api.query(buckets_query)
        
        print(f"Query executed successfully. Result type: {type(result)}")
        
        # List buckets using API
        buckets_api = client.buckets_api()
        buckets = buckets_api.find_buckets()
        print(f"Available buckets: {[b.name for b in buckets.buckets]}")
        
    client.close()
    
except Exception as e:
    print(f"Connection failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Install required packages
import subprocess
import sys

print("Installing required InfluxDB packages...")

# Install influxdb-client for v2
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "influxdb-client==1.49.0", "--quiet"])
    print("✓ influxdb-client installed")
except Exception as e:
    print(f"Failed to install influxdb-client: {e}")

# Restart kernel to reload imports
print("Packages installed. Please restart the kernel if needed.")

In [ ]:
# Test InfluxDB v2 with installed packages
print("Testing InfluxDB v2 connection with installed packages...")

try:
    import influxdb_client
    from influxdb_client import InfluxDBClient
    print(f"✓ InfluxDB client version: {influxdb_client.__version__}")
    
    # Connection parameters
    url = "http://influxdb:8086"
    token = "qtab_factor_token_2024_secure"
    org = "qtab"
    
    print(f"Connecting to {url}")
    
    # Create client
    client = InfluxDBClient(url=url, token=token, org=org, timeout=30000)
    
    # Test health
    health = client.health()
    print(f"Health status: {health.status}")
    print(f"Health message: {health.message}")
    
    if health.status == "pass":
        print("✓ InfluxDB v2 connection successful!")
        
        # List buckets
        buckets_api = client.buckets_api()
        buckets = buckets_api.find_buckets()
        bucket_names = [b.name for b in buckets.buckets if not b.name.startswith('_')]
        print(f"Available data buckets: {bucket_names}")
        
        # Test if our target bucket exists
        if 'jkp_factors' in bucket_names:
            print("✓ Target bucket 'jkp_factors' found")
            
            # Try a simple query
            query_api = client.query_api()
            simple_query = '''
            from(bucket: "jkp_factors")
                |> range(start: -7d)
                |> limit(n: 5)
            '''
            
            print("Testing simple data query...")
            result = query_api.query(simple_query)
            
            record_count = 0
            for table in result:
                for record in table.records:
                    record_count += 1
                    if record_count <= 2:  # Show first 2 records
                        print(f"  Sample record: {record.get_time()} - {record.get_measurement()} - {record.get_field()}: {record.get_value()}")
            
            print(f"✓ Query successful: found {record_count} records in last 7 days")
        else:
            print("⚠ Target bucket 'jkp_factors' not found")
            print("Available buckets:", bucket_names)
    
    client.close()
    
except Exception as e:
    print(f"Connection/query failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Test InfluxDB connection with host fallback
print("Testing InfluxDB connection with host fallback...")

import socket
from influxdb_client import InfluxDBClient

# Connection parameters
token = "qtab_factor_token_2024_secure"
org = "qtab"

# Host candidates to try
host_candidates = [
    "influxdb",
    "localhost", 
    "127.0.0.1",
    "host.docker.internal"
]

for host in host_candidates:
    url = f"http://{host}:8086"
    print(f"\nTrying {url}...")
    
    # Test socket connection first
    try:
        with socket.create_connection((host, 8086), timeout=5):
            print(f"  ✓ Socket connection to {host}:8086 successful")
            
            # Test InfluxDB connection
            try:
                client = InfluxDBClient(url=url, token=token, org=org, timeout=10000)
                health = client.health()
                
                print(f"  Health status: {health.status}")
                
                if health.status == "pass":
                    print(f"  ✓ InfluxDB connection successful to {host}!")
                    
                    # List buckets
                    buckets_api = client.buckets_api()
                    buckets = buckets_api.find_buckets()
                    bucket_names = [b.name for b in buckets.buckets if not b.name.startswith('_')]
                    print(f"  Available buckets: {bucket_names}")
                    
                    # Test query if jkp_factors exists
                    if 'jkp_factors' in bucket_names:
                        print("  ✓ Found 'jkp_factors' bucket - testing query...")
                        
                        query_api = client.query_api()
                        test_query = '''
                        from(bucket: "jkp_factors")
                            |> range(start: -30d)
                            |> limit(n: 3)
                        '''
                        
                        result = query_api.query(test_query)
                        
                        record_count = 0
                        for table in result:
                            for record in table.records:
                                record_count += 1
                                break  # Just count, don't print all
                        
                        print(f"  ✓ Query successful: {record_count} records found")
                        
                        if record_count > 0:
                            print(f"  🎉 SUCCESS: Working connection to {host}:8086")
                            client.close()
                            break
                    
                client.close()
                
            except Exception as e:
                print(f"  ✗ InfluxDB connection failed: {e}")
                
    except (socket.error, socket.timeout) as e:
        print(f"  ✗ Socket connection to {host}:8086 failed: {e}")

print("\nHost testing complete.")

In [ ]:
# Test Enhanced Barra Loader with working localhost connection
print("Testing Enhanced Barra Loader with localhost connection...")

# Add src to path and reload
import sys
import importlib
sys.path.append('/home/jovyan/work/src')

try:
    import barra_loader
    importlib.reload(barra_loader)
    from barra_loader import EnhancedBarraLoader
except Exception as e:
    print(f"Import error: {e}")
    print("Module might need to be reloaded or path issue")
    # Try direct execution of the module
    exec(open('/home/jovyan/work/src/barra_loader.py').read())

# Create configuration for localhost
config = {
    'influx_host': 'localhost',
    'influx_port': 8086,
    'influx_token': 'qtab_factor_token_2024_secure',
    'influx_org': 'qtab',
    'source_database': 'jkp_factors',
    'barra_database': 'barra_models',
    'timeout': 30,
    'max_retries': 3
}

print("Initializing Enhanced Barra Loader with localhost configuration...")

try:
    loader = EnhancedBarraLoader(config)

    # Check status
    status = loader.get_connection_status()
    print("\nConnection Status:")
    for key, value in status.items():
        print(f"  {key}: {value}")

    # Test data loading capabilities
    if status.get('connection_working', False) or status.get('client_initialized', False):
        print(f"\n✓ InfluxDB version: {loader.influx_version}")
        print("✓ Connection working!")
        
        # Test checking existing data
        print("\nTesting existing data checks...")
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
        
        existing_check = loader.check_existing_data('jkp_factors', start_date, end_date)
        print(f"Existing data in jkp_factors: {existing_check}")
        
        existing_barra = loader.check_existing_data('barra_models', start_date, end_date)
        print(f"Existing Barra models: {existing_barra}")
        
        # Test model dates
        model_dates = loader.get_model_dates(start_date, end_date)
        print(f"Available model dates: {len(model_dates)}")
        
        # Test data loading (even if empty)
        print(f"\nTesting data loading from {start_date} to {end_date}...")
        data = loader.load_factor_data(start_date, end_date)
        print(f"Data loading result: {len(data)} records")
        
        if not data.empty:
            print("Sample data:")
            print(data.head())
        else:
            print("No data found in the specified date range (this is expected if bucket is empty)")
        
        print("\n🎉 Enhanced Barra Loader is working correctly!")
        
    else:
        print("✗ Connection issues persist")
        
except Exception as e:
    print(f"Error with Enhanced Barra Loader: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Direct test of Barra Loader functionality
print("Direct test of Enhanced Barra Loader functionality...")

# Test InfluxDB v2 connection directly
from influxdb_client import InfluxDBClient

# Working configuration
config = {
    'url': 'http://localhost:8086',
    'token': 'qtab_factor_token_2024_secure',
    'org': 'qtab',
    'source_bucket': 'jkp_factors',
    'barra_bucket': 'barra_models'
}

print("Testing direct InfluxDB v2 operations...")

try:
    # Create client
    client = InfluxDBClient(
        url=config['url'],
        token=config['token'],
        org=config['org'],
        timeout=30000
    )
    
    # Test connection
    health = client.health()
    print(f"✓ Health: {health.status}")
    
    if health.status == "pass":
        # Test bucket access
        buckets_api = client.buckets_api()
        buckets = buckets_api.find_buckets()
        bucket_names = [b.name for b in buckets.buckets if not b.name.startswith('_')]
        print(f"✓ Available buckets: {bucket_names}")
        
        # Test if we can create the barra_models bucket if it doesn't exist
        if config['barra_bucket'] not in bucket_names:
            print(f"Creating {config['barra_bucket']} bucket...")
            try:
                # Get the organization
                orgs_api = client.organizations_api()
                orgs = orgs_api.find_organizations()
                org_id = None
                for org in orgs:
                    if org.name == config['org']:
                        org_id = org.id
                        break
                
                if org_id:
                    # Create bucket
                    bucket = buckets_api.create_bucket(
                        bucket_name=config['barra_bucket'],
                        org_id=org_id,
                        retention_rules=[]
                    )
                    print(f"✓ Created bucket: {bucket.name}")
                else:
                    print("✗ Could not find organization")
            except Exception as e:
                print(f"⚠ Could not create bucket: {e}")
        
        # Test basic query capability
        if config['source_bucket'] in bucket_names:
            print(f"Testing query on {config['source_bucket']}...")
            
            query_api = client.query_api()
            test_query = f'''
            from(bucket: "{config['source_bucket']}")
                |> range(start: -90d)
                |> limit(n: 1)
            '''
            
            try:
                result = query_api.query(test_query)
                record_count = sum(len(table.records) for table in result)
                print(f"✓ Query test successful: {record_count} records found")
                
                if record_count == 0:
                    print("⚠ No data in source bucket - you may need to load factor data first")
                else:
                    print("✓ Data available for Barra model building")
                    
            except Exception as e:
                print(f"✗ Query failed: {e}")
        
        # Test write capability
        print("Testing write capability...")
        try:
            from influxdb_client import Point, WritePrecision
            
            write_api = client.write_api()
            
            # Test point
            test_point = Point("test_measurement") \
                .tag("test_tag", "test_value") \
                .field("test_field", 123.45) \
                .time(datetime.now(), WritePrecision.NS)
            
            write_api.write(bucket=config['source_bucket'], record=test_point)
            print("✓ Write test successful")
            
        except Exception as e:
            print(f"⚠ Write test failed: {e}")
        
        print(f"\n🎉 InfluxDB v2 setup is working correctly!")
        print(f"   - Host: {config['url']}")
        print(f"   - Source bucket: {config['source_bucket']}")
        print(f"   - Barra bucket: {config['barra_bucket']}")
        print(f"   - Ready for Barra model implementation")
        
    client.close()
    
except Exception as e:
    print(f"✗ Connection failed: {e}")
    import traceback
    traceback.print_exc()

## ✅ SOLUTION SUMMARY

### Issues Resolved:

1. **DNS Resolution Fixed**: 
   - ❌ `influxdb:8086` was failing with DNS resolution errors
   - ✅ `localhost:8086` works correctly from Jupyter container

2. **Protocol Mismatch Fixed**:
   - ❌ InfluxDB v3 client was incompatible with InfluxDB v2.7.12 server
   - ✅ InfluxDB v2 client (`influxdb-client==1.49.0`) works properly

3. **Database Setup Complete**:
   - ✅ `jkp_factors` bucket exists (source data)
   - ✅ `barra_models` bucket created (for Barra results)
   - ✅ Query and write operations working

### Working Configuration:
```python
config = {
    'url': 'http://localhost:8086',
    'token': 'qtab_factor_token_2024_secure',
    'org': 'qtab',
    'source_bucket': 'jkp_factors',
    'barra_bucket': 'barra_models'
}
```

### Next Steps:

1. **Load Factor Data**: The `jkp_factors` bucket is empty - need to populate with JKP characteristics data
2. **Update Enhanced Barra Loader**: Modify to use `localhost:8086` as default
3. **Integrate with Barra Model**: Use the working connection in your main Barra implementation
4. **Test Full Pipeline**: Once data is loaded, test complete Barra model building

### Files Updated:
- ✅ `docker-compose.yml` - networking improvements
- ✅ `src/barra_loader.py` - enhanced loader with v2 support
- ✅ `Enhanced_Barra_Loader_Guide.md` - comprehensive documentation

In [ ]:
# Test data loading with recent data
if status.get('connection_working', False) or status.get('client_initialized', False):
    print("Testing data loading...")
    
    # Define date range for testing (last 30 days)
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
    
    print(f"Loading data from {start_date} to {end_date}")
    
    # Load factor data
    factor_data = loader.load_factor_data(start_date, end_date)
    
    print(f"\nLoaded {len(factor_data)} records")
    
    if not factor_data.empty:
        print("\nData preview:")
        print(factor_data.head())
        print("\nData columns:")
        print(factor_data.columns.tolist())
        print("\nData shape:", factor_data.shape)
    else:
        print("No data loaded - checking connectivity issues")
else:
    print("Connection not available for testing")

In [ ]:
# Check for existing Barra model data
print("Checking for existing Barra model data...")

# Check last 90 days
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')

existing_data = loader.check_existing_data('barra_models', start_date, end_date)
print(f"\nExisting data check for {start_date} to {end_date}:")
for key, value in existing_data.items():
    print(f"  {key}: {value}")

# Get available model dates
model_dates = loader.get_model_dates(start_date, end_date)
print(f"\nAvailable model dates: {len(model_dates)}")
if model_dates:
    print(f"Date range: {min(model_dates)} to {max(model_dates)}")
    print(f"Recent dates: {model_dates[-5:] if len(model_dates) > 5 else model_dates}")

In [ ]:
# Test model update functionality
print("Testing model update functionality...")

# Use a small date range for testing
test_end_date = datetime.now().strftime('%Y-%m-%d')
test_start_date = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')

print(f"Testing update for {test_start_date} to {test_end_date}")

# Test update (this will check data availability but not run full model yet)
update_result = loader.update_model(test_start_date, test_end_date, force_rebuild=False)

print("\nUpdate result:")
for key, value in update_result.items():
    print(f"  {key}: {value}")

In [ ]:
# Network diagnostics
print("Running network diagnostics...")

import socket

# Test different host connections
hosts_to_test = [
    ('influxdb', 8086),
    ('localhost', 8086),
    ('127.0.0.1', 8086),
    ('host.docker.internal', 8086)
]

print("\nTesting host connectivity:")
for host, port in hosts_to_test:
    try:
        with socket.create_connection((host, port), timeout=5):
            print(f"  ✓ {host}:{port} - REACHABLE")
    except (socket.error, socket.timeout) as e:
        print(f"  ✗ {host}:{port} - FAILED ({e})")

# Check environment variables
print("\nEnvironment variables:")
env_vars = ['INFLUXDB_HOST', 'INFLUXDB_PORT', 'INFLUXDB_TOKEN', 'INFLUXDB_ORG', 'INFLUXDB_BUCKET']
for var in env_vars:
    value = os.getenv(var, 'NOT SET')
    print(f"  {var}: {value}")

## Next Steps

1. **If connectivity works**: Proceed with full Barra model implementation
2. **If connectivity fails**: Check Docker networking and InfluxDB container status
3. **Model Development**: Integrate the enhanced loader with the main Barra model logic
4. **Maintenance**: Use the update_model functionality for incremental updates